# Clase 11 — Pandas: Series y DataFrame

## Objetivos de la clase

- Entender qué es una **Series** y qué es un **DataFrame**, y en qué se diferencian.
- Crear un DataFrame a partir de un diccionario, y guardarlo/cargarlo con `to_csv()` / `read_csv()`.
- Seleccionar columnas con corchete simple (da una **Series**) vs corchete doble (da un **DataFrame**).
- Dominar la diferencia entre `.loc` (por etiqueta) y `.iloc` (por posición), sobre todo en el slicing.
- Hacer el "primer contacto" con un DataFrame nuevo: `head()`, `info()`, `describe()`, `shape`.

Vamos a trabajar con una versión del dataset **TopSellingAlbums** (los discos más vendidos de la historia: Michael Jackson, AC/DC, Pink Floyd...). En vez de descargarlo de internet, lo vamos a **crear directamente en el notebook** con un diccionario, para no depender de ninguna conexión.


## 1. ¿Qué es una Series?

Una **Series** es un array de una sola dimensión, con una diferencia clave respecto a una lista de Python: cada valor tiene una etiqueta asociada, llamada **índice**. Piénsala como una columna de hoja de cálculo con su propia numeración de filas pegada al lado.

Si no le das índices propios, pandas asigna automáticamente `0, 1, 2, 3, 4...`.


In [ ]:
import pandas as pd

# Crear una Series a partir de una lista
ventas = [46, 26.1, 24.2, 26.1, 20.6]
s = pd.Series(ventas)
print(s)


Con un índice por defecto (0,1,2,3...), acceder por **etiqueta** y por **posición** da el mismo resultado — esa coincidencia es justo la trampa que hay que aprender a evitar (más adelante, con `.loc`/`.iloc` sobre un DataFrame, vas a ver por qué importa la diferencia).


In [ ]:
print(s[2])         # Accede por etiqueta (índice 2) -> 24.2
print(s.iloc[3])    # Accede por posición (posición 3) -> 26.1
print(s[1:4])        # Rango de elementos por etiqueta


Una Series trae atributos y métodos incorporados para explorar y resumir los datos sin escribir un bucle. Guarda el nombre `apply()`: es tu herramienta para aplicar cualquier transformación personalizada a cada valor.


In [ ]:
print(s.values)        # el contenido como array de NumPy
print(s.index)         # las etiquetas del índice
print(s.shape)         # dimensiones, por ejemplo (5,)
print(s.mean())        # promedio
print(s.sum())         # suma total
print(s.unique())      # valores distintos, sin repetir
print(s.sort_values()) # ordena la Series por sus valores
print(s.isnull())      # True/False según si cada valor es nulo (NaN)
print(s.apply(lambda x: x * 1_000_000))  # aplica una función a cada elemento


## 2. ¿Qué es un DataFrame?

Si una Series es una columna, un **DataFrame** es la tabla completa: una estructura de dos dimensiones, con filas y columnas etiquetadas, donde cada columna puede tener su propio tipo de dato. Es el objeto que más vas a manipular en cualquier proyecto de datos con Python.

La forma más directa de crear uno a mano es a partir de un **diccionario**: cada clave se convierte en el nombre de una columna, y cada valor (una lista) se convierte en los datos de esa columna, fila por fila.


In [ ]:
datos = {
    'Artist': ['Michael Jackson', 'AC/DC', 'Pink Floyd'],
    'Album': ['Thriller', 'Back in Black', 'The Dark Side of the Moon'],
    'Released': [1982, 1980, 1973],
}
df_mini = pd.DataFrame(datos)
df_mini


Pandas alinea automáticamente cada lista como una columna y numera las filas empezando en 0. Ese número es el índice del DataFrame, y por defecto es puramente posicional — más abajo vas a ver cómo cambiarlo con `set_index()`.


## 3. Cargar datos reales: `read_csv()` y `read_excel()`

En la práctica, casi nunca vas a escribir un DataFrame a mano: vas a cargarlo desde un archivo, con una función por formato:

```python
df = pd.read_csv('TopSellingAlbums.csv')
df = pd.read_excel('TopSellingAlbums.xlsx')
```

En ambos casos el resultado es el mismo tipo de objeto: un DataFrame.

**Nota:** para no depender de descargar un archivo desde internet (puede fallar sin conexión), vamos a **crear** el DataFrame completo con un diccionario — reproduciendo los datos reales de `TopSellingAlbums` — y luego lo guardamos y lo volvemos a leer con `to_csv()` / `read_csv()`, para practicar el flujo real de lectura de un CSV sin depender de la red.


In [ ]:
datos_albumes = {
    'Artist': ['Michael Jackson', 'AC/DC', 'Pink Floyd', 'Whitney Houston',
               'Meat Loaf', 'Eagles', 'Bee Gees', 'Fleetwood Mac'],
    'Album': ['Thriller', 'Back in Black', 'The Dark Side of the Moon', 'The Bodyguard',
              'Bat Out of Hell', 'Their Greatest Hits (1971-1975)', 'Saturday Night Fever', 'Rumours'],
    'Released': [1982, 1980, 1973, 1992, 1977, 1976, 1977, 1977],
    'Length': ['00:42:19', '00:42:11', '00:42:49', '00:57:44',
               '00:46:33', '00:43:08', '01:15:54', '00:40:01'],
    'Genre': ['Pop, rock, R&B', 'Hard rock', 'Progressive rock', 'Soundtrack/R&B, soul, pop',
              'Hard rock, progressive rock', 'Rock, soft rock, folk rock', 'Disco', 'Soft rock'],
    'Sales': [46, 26.1, 24.2, 26.1, 20.6, 32.2, 20.6, 27.9],
    'Rating': [10.0, 8.5, 9.5, 7.0, 7.0, 9.5, 9.0, 9.5],
}

df = pd.DataFrame(datos_albumes)
df


Ahora simulamos el flujo real: guardamos el DataFrame como un archivo `.csv` en disco, y lo volvemos a cargar con `read_csv()`, exactamente como harías con un archivo que te hubieran entregado.


In [ ]:
# Guardamos el DataFrame como si fuera el archivo que "recibimos" de otra fuente
df.to_csv('TopSellingAlbums.csv', index=False)

# ...y lo volvemos a cargar con read_csv(), como harías con cualquier archivo real
df = pd.read_csv('TopSellingAlbums.csv')
df

# Si el archivo fuera un Excel, el patrón es idéntico (requiere el paquete openpyxl):
# df = pd.read_excel('TopSellingAlbums.xlsx')


El primer comando que deberías escribir después de cualquier `read_csv()` es `head()`: te confirma que la carga funcionó y te da una idea inmediata de cómo lucen los datos.


In [ ]:
df.head()      # primeras 5 filas


In [ ]:
df.head(3)     # primeras 3 filas, si quieres menos


In [ ]:
df.tail(2)     # las últimas 2 filas


## 4. Seleccionar columnas: ¿Series o DataFrame?

Aquí aparece la primera fuente de confusión, y vale la pena interiorizarla ahora: **la cantidad de corchetes cambia el tipo de resultado.**

- `df['col']` (un corchete) → una **Series**.
- `df[['col']]` (corchetes dobles, con una lista adentro) → un **DataFrame** de una sola columna.
- `df[['col1', 'col2']]` (lista con varios nombres) → un **DataFrame** con esas columnas, en el orden pedido.


In [ ]:
x = df['Length']
print(type(x))   # <class 'pandas.core.series.Series'>
x


In [ ]:
x = df[['Length']]
print(type(x))   # <class 'pandas.core.frame.DataFrame'>
x


In [ ]:
y = df[['Artist', 'Length', 'Genre']]
y


¿Por qué importa la diferencia? Porque algunos métodos esperan una Series y otros un DataFrame, y un error de tipo aquí suele venir de haber usado un corchete de más o de menos sin darte cuenta. Cuando dudes, usa `type()`.


## 5. `.loc` vs `.iloc`: la diferencia que confunde a todo principiante

Pandas te da dos formas de acceder a filas (y celdas concretas) de un DataFrame:

- **`.iloc`** selecciona por **posición entera**: "dame la fila número 2", sin importar cómo se llame. Funciona como el índice de una lista de Python.
- **`.loc`** selecciona por **etiqueta**: "dame la fila llamada 2" o "la fila llamada 'Jane'", sin importar en qué posición física esté.

| | `.iloc` (por posición) | `.loc` (por etiqueta) |
|---|---|---|
| Qué espera | Números enteros de posición | Nombres de fila/columna (o números, si son las etiquetas del índice) |
| Límite superior del rango | **Exclusivo** — no incluye el último | **Inclusivo** — sí incluye el último |
| Ejemplo de filas | `df.iloc[0:2]` → filas 0 y 1 | `df.loc[0:2]` → filas 0, 1 **y** 2 |

Con un índice recién cargado (0,1,2,3... en orden), ambos dan el mismo resultado para **una sola celda** — ahí nace la confusión, porque parecen intercambiables:


In [ ]:
print(df.iloc[0, 0])          # Valor en la fila 0, columna 0 (por posición)
print(df.loc[0, 'Artist'])    # Valor en la fila con etiqueta 0, columna 'Artist' (por nombre)
# Con el índice por defecto, ambos devuelven "Michael Jackson"

print(df.iloc[1, 2])          # Fila 1, columna 2 (por posición) -> 1980
print(df.loc[1, 'Released'])  # Fila con etiqueta 1, columna 'Released' -> 1980


Donde de verdad se separan es en el **slicing** (los rangos con `:`). Compáralo sobre el mismo DataFrame, seleccionando filas y columnas a la vez:


In [ ]:
# Con iloc: filas 0 y 1 (la 2 queda fuera), columnas 0, 1 y 2 (la 3 queda fuera)
df.iloc[0:2, 0:3]


In [ ]:
# Con loc: filas con etiqueta 0, 1 Y 2 (el límite superior SÍ se incluye)
# columnas desde 'Artist' hasta 'Released' incluida
df.loc[0:2, 'Artist':'Released']


In [ ]:
print("Filas con df.iloc[0:2]:", len(df.iloc[0:2]))
print("Filas con df.loc[0:2]: ", len(df.loc[0:2]))
# Mismo rango escrito "0 a 2", cantidad de filas distinta


La diferencia se vuelve todavía más visible en cuanto el índice deja de ser una simple secuencia de números. Puedes reemplazar el índice numérico por etiquetas propias con `set_index()`:


In [ ]:
df2 = df.set_index('Artist')   # ahora el índice son los nombres de artista, no 0,1,2...
df2.head()


In [ ]:
print(df2.loc['Michael Jackson', 'Released'])   # por etiqueta: funciona, busca el nombre
print(df2.iloc[0, 1])                            # por posición: sigue funcionando igual que antes


Una vez que el índice son nombres, `.iloc` ni se entera del cambio — sigue contando posiciones desde 0 — mientras que `.loc` ahora exige las etiquetas nuevas, no los números viejos. Si intentas `df2.loc[0, 'Released']`, vas a obtener un error, porque `0` ya no es una etiqueta válida del índice:


In [ ]:
try:
    df2.loc[0, 'Released']
except KeyError as e:
    print("Error esperado, porque 0 ya no es una etiqueta del índice de df2:")
    print(e)


**Regla práctica para no confundirte nunca más:** `.iloc` cuenta como una lista de Python (el límite de arriba se queda fuera). `.loc` busca como en un diccionario o un índice de libro (el límite de arriba se incluye, porque es un nombre, no un conteo).


## 6. Exploración rápida: `head()`, `info()`, `describe()`, `shape`

Antes de filtrar, agrupar o graficar nada, tu primer minuto con cualquier DataFrame nuevo debería ir en estos comandos:

- `head()` / `tail()`: primeras/últimas filas — ¿cómo lucen los datos?
- `info()`: cuántas filas y columnas hay, el tipo de dato de cada columna y cuántos valores no nulos tiene cada una.
- `describe()`: conteo, media, desviación estándar, mínimo, máximo y cuartiles de cada columna **numérica** (fíjate que `Length`, que es texto, no aparece aquí).
- `shape`: una tupla `(filas, columnas)` con el tamaño total.


In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
df.shape


## Resumen: lo que te llevas de esta clase

- Una **Series** es un array de una dimensión con un índice etiquetado; una **columna** de una tabla, con memoria de sus propias etiquetas.
- Un **DataFrame** es una tabla de dos dimensiones, con filas y columnas etiquetadas — el objeto central de pandas. Se crea desde un diccionario (`pd.DataFrame()`) o, mucho más a menudo, desde un archivo (`pd.read_csv()`, `pd.read_excel()`).
- `df['col']` (un corchete) devuelve una **Series**; `df[['col']]` o `df[['col1','col2']]` (corchetes dobles) devuelve un **DataFrame**. Usa `type()` cuando dudes.
- **`.iloc[fila, columna]`** selecciona por **posición entera**, igual que una lista: el límite superior de un rango queda **excluido**.
- **`.loc[fila, columna]`** selecciona por **etiqueta** (nombre): el límite superior de un rango queda **incluido**. Si cambias el índice con `set_index()`, `.loc` debe usar las nuevas etiquetas.
- `df.iloc[0:2, 0:3]` y `df.loc[0:2, 'A':'C']` no son intercambiables aunque se vean parecidos: revisa siempre si el límite de arriba entra o no en el resultado.
- `head()`, `tail()`, `info()`, `describe()` y `shape` son tu ritual de los primeros 60 segundos con cualquier DataFrame nuevo.
